# Digit Recognition — Why the Custom CNN and What It Cost Us

This notebook walks through **why** the OCR layer is a 102,026-parameter custom CNN (not Tesseract, not a pretrained model) and **how** we got to **66.6% filled-cell / 98.4% empty-cell / 81.3% overall** real-photo accuracy on the 38-image ground-truth set from a 99.5% synthetic test-split score. The current production checkpoint is **v5.1**, which uses class-weighted cross-entropy loss to balance the empty-cell class against the nine digit classes. The real story is several bad ideas, one data-leakage incident, and multiple retrain-then-rollback cycles that taught us where the actual bottleneck lives.

**Prerequisites.** Full dev requirements (`pip install -r ../requirements.txt`) — this notebook needs PyTorch for architecture introspection and dataset sampling. The trained checkpoint at `../app/ml/checkpoints/sudoku_cnn.onnx` (+ its `.data` sidecar) is needed to reproduce the real-photo accuracy numbers.

**Related reading**

- [`ocr_analysis.ipynb`](ocr_analysis.ipynb) — the deep per-cell error-taxonomy analysis that drove the decision to stop improving the model
- `../app/ml/model.py:24` — `SudokuCNN` architecture
- `../app/ml/dataset.py` — `PrintedDigitDataset`, `EmptyCellDataset`, `Chars74KFontDataset`, `create_datasets`
- `../app/ml/recognizer.py` — the ONNX Runtime wrapper used at inference time

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))


## 1. The architecture

Three `Conv → BatchNorm → ReLU → MaxPool` blocks with channel expansion `1 → 32 → 64 → 128`, then `AdaptiveAvgPool2d(1)` (global pooling) and a small `Linear(128→64) → ReLU → Dropout(0.3) → Linear(64→10)` classifier head. Input is 28×28 grayscale; output is 10 logits (class 0 = empty, 1–9 = digits).

Why "custom" and not a pretrained model? The task is *tiny* — 10 classes at 28×28 is barely harder than MNIST. Pretrained image classifiers (ResNet, EfficientNet, MobileNet) are designed for 224×224 RGB inputs with 1000+ classes; loading one would add tens of megabytes of weights for no accuracy gain, and the smaller pretrained digit models I found are trained on different distributions (Chars74K, SVHN) that don't match newsprint. The custom CNN is ~400 KB deployed (ONNX header + external weight sidecar) and large enough to hit 99.5% on a font-disjoint held-out split (MNIST labels 1-9 + Chars74K + empty cells). The bottleneck is NOT model capacity, as the deep error analysis in `ocr_analysis.ipynb` shows — see section 6 below.

In [ ]:
import torch
from app.ml.model import SudokuCNN

model = SudokuCNN()
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameter count: {n_params:,}")
print()
print("Per-layer breakdown:")
for name, p in model.named_parameters():
    print(f"  {name:<30} {tuple(p.shape)!s:<20} {p.numel():>8}")


## 2. Training data — four sources, class-weighted loss, one lesson

The training set is `ConcatDataset([MNIST 1-9, EmptyCellDataset, PrintedDigitDataset, Chars74KFontDataset])` — see `app/ml/dataset.py::create_datasets`. Why four sources?

- **MNIST labels 1-9 only (~54k samples)** — the handwritten-digit baseline. MNIST label 0 is **dropped** via `_load_mnist_no_zero` because class 0 in this project means "empty cell", not "digit zero". Including MNIST's ~5,923 handwritten '0' glyphs would pollute the empty-cell label with round digit-shaped images — a semantic contradiction with the production inference contract. This was a v4.2 fix; prior versions included MNIST zeros and suffered from higher hallucination rates.
- **`PrintedDigitDataset` (~4,500 samples)** — 9 digits × 500 samples, rendered from **67 allowlist-validated Latin-digit system fonts** (narrowed from ~371 discovered fonts via a two-stage filter: an explicit `LATIN_FONT_ALLOWLIST` match, then `_font_has_distinct_latin_digits` which rasterises all ten digits on a **56×56** intermediate canvas with `textbbox`-measured centering and rejects any font whose pairwise pixel correlations exceed `max > 0.995 OR mean > 0.85`). Augmented with random ±10° rotation, additive noise, and Gaussian blur. This is the newsprint-lookalike source.
- **`Chars74KFontDataset` (~7.3k train / ~1.8k test)** — printed digits 1-9 rendered from ~800 computer fonts, with a **font-disjoint train/test split** (~600 train fonts, ~200 held-out test fonts). This makes the synthetic test accuracy a meaningful generalisation number — the test set evaluates on fonts the model has never seen, not just held-out samples from training fonts.
- **`EmptyCellDataset` (5,000 samples)** — four synthetic variants for the "class 0 = empty" label: blank cells, uniform-noise cells, linear-gradient cells, and border-stroke cells — **rewritten in v4.2 to match measured GT empty-cell statistics** (mean ~56, std ~8, 5th-percentile ~44). Without this, the CNN would have no "empty" class to learn and every cell would be forced into 1–9.

**Class-weighted loss.** Dropping MNIST zeros cut class 0's share of the training pool from ~14% to ~7%. To compensate, v5.1 trains with `CrossEntropyLoss(weight=[2.0, 1.0, 1.0, ..., 1.0])`, doubling class 0's gradient contribution. This restores the empty-cell detection rate without inflating the dataset. The effect is dramatic: without the weight, the model hallucinated 222 empty cells as digits; with it, hallucinations dropped to **21**.

**The test split includes Chars74K held-out fonts, MNIST test (labels 1-9), and EmptyCellDataset test.** This was a methodological fix from the 2026-04-10 redesign — prior versions tested only on MNIST + empty cells, so "synthetic test accuracy" was measuring MNIST recall, not generalisation to unseen printed fonts.

In [ ]:
# Visualize one sample from each of the four training sources
from app.ml.dataset import PrintedDigitDataset, EmptyCellDataset, Chars74KFontDataset
from torchvision import datasets as tv_datasets
from torchvision import transforms as tv_transforms

mnist = tv_datasets.MNIST(root=str(PROJECT_ROOT / "data" / "mnist"), train=True,
                          download=True, transform=tv_transforms.ToTensor())
printed = PrintedDigitDataset(count_per_digit=100, seed=42)
chars74k = Chars74KFontDataset(split="train")
empty = EmptyCellDataset(count=100, seed=42)

fig, axes = plt.subplots(4, 9, figsize=(14, 7))
for i in range(9):
    # MNIST — skip label 0 (class 0 = empty, not digit zero)
    idx = next(k for k in range(len(mnist)) if mnist[k][1] == i + 1)
    axes[0, i].imshow(mnist[idx][0].squeeze(), cmap="gray")
    axes[0, i].set_title(f"MNIST\nlabel {i+1}", fontsize=8); axes[0, i].axis("off")

    axes[1, i].imshow(printed[i * (len(printed) // 9)][0].squeeze(), cmap="gray")
    axes[1, i].set_title("Printed\n(67 fonts)", fontsize=8); axes[1, i].axis("off")

    # Chars74K — find a sample for digit i+1
    c74k_idx = next((k for k in range(len(chars74k)) if chars74k[k][1] == i + 1), 0)
    axes[2, i].imshow(chars74k[c74k_idx][0].squeeze(), cmap="gray")
    axes[2, i].set_title("Chars74K\n(font-disjoint)", fontsize=8); axes[2, i].axis("off")

    axes[3, i].imshow(empty[i * (len(empty) // 9)][0].squeeze(), cmap="gray")
    axes[3, i].set_title("Empty", fontsize=8); axes[3, i].axis("off")

plt.suptitle("Training data sources (v5.1)", fontsize=12)
plt.tight_layout(); plt.show()

## 3. The train/inference preprocessing mismatch that cost 5 points

The CNN was trained on grayscale MNIST tensors in `[0, 1]` continuous. The inference path was applying Otsu binarization, forcing pixels to `{0, 1}`. The network was seeing a distribution it had never been trained on.

Fixing the mismatch — matching inference preprocessing to training preprocessing (grayscale normalisation instead of binarization) — bumped overall OCR accuracy from **57.3% to 62.2%** in a single commit (`f49b459`, 2026-04-03). Five points of free accuracy from deleting a call to `cv2.threshold`. The kind of bug you find by printing one training sample and one inference sample side by side and noticing they look different — which is exactly what I should have done at any point before the cleanup.

## 4. The data-leakage incident

My first attempt at fixing the 61% number was to add the 3,078 ground-truth cells directly to the training set. The next run of `evaluate_ocr.py` reported **87.8% overall and 1 hallucination**. That looked incredible. I almost shipped it.

It was textbook data leakage: the network had seen every cell it was being graded on, and `evaluate_ocr.py` was effectively a train-accuracy report. The honest fix was to replace the leaked cells with `PrintedDigitDataset` (the ~4,500 rendered-font samples described above) and accept the resulting 76.3% overall + 86 hallucinations as the real baseline. The leaked 87.8% would have looked better on a CV and told me nothing about generalization.

**Before (MNIST-only, Otsu binarization):** 57.3% overall, 594 hallucinations.
**After leaked fix (MNIST + GT cells in train):** 87.8% overall, 1 hallucination. *Unshippable — data leakage.*
**After honest fix (MNIST + printed + empty, grayscale preproc):** 76.3% overall, 86 hallucinations. *Current baseline.*

The filled-cell accuracy hardly changed (60.6% → 61.6%) because the conservative confidence threshold (0.85) turns CNN uncertainty into *misses* rather than *wrong digits* — the failure mode moved but the total filled count didn't. See section 6 for why filled-cell accuracy is stuck here.


## 5. Synthetic vs real — the error taxonomy

On a held-out test split (MNIST 1-9 + Chars74K held-out fonts + empty cells) the CNN reaches **99.5% test accuracy** (see `app/ml/checkpoints/training_results.json`). On real photographed cells from the 38-image GT it reaches **66.6% filled / 98.4% empty / 81.3% overall**. Where does the 33-point filled-cell gap go?

The full breakdown is in [`ocr_analysis.ipynb`](ocr_analysis.ipynb), which classifies every single filled-cell prediction into one of four buckets:

| Bucket | Count | % of filled | Interpretation |
|---|---|---|---|
| Correct (CNN + empty filter agree) | 922 | 62.4% | The pipeline works on clean cells |
| **Missed by low confidence** | 337 | 22.8% | CNN refuses to commit (softmax below 0.50) — dominant error mode |
| Hallucinated (empty → digit) | 21 | — | CNN labels an empty cell as 1–9 |
| Wrong digit (filled → wrong) | 219 | **14.8%** | CNN commits to the wrong digit |

**v5.1's key trade-off: when it commits, it's more often right.** Compared to the v3 baseline, v5.1 trades +87 more misses for -73 fewer wrong digits and -19 fewer hallucinations. The model is more conservative — it abstains more often, but when it does commit, the prediction is more reliable. This is the desired behaviour for Sudoku OCR because:

1. A missed digit is recoverable via the confidence-colored review UI
2. A wrong digit corrupts the solver input silently
3. A hallucination breaks solvability entirely (creates duplicates or dead-end grids)

Hallucinations cut from 40 (v3) to **21** (v5.1) is the headline win — those 19 fewer false digits directly translate to fewer unsolvable puzzles.

## 6. Why input quality is the bottleneck — and why a bigger CNN made it worse

**The confidence threshold story.** v5.1 ships with a confidence threshold of **0.50** (up from 0.10 in v3). The class-weighted loss produces a cleaner class 0 boundary, which makes the softmax distributions more peaked — the model is more decisive. This lets us raise the threshold without losing too many correct predictions. At threshold 0.10, v5.1 would hallucinate 222 cells; at 0.50, it hallucinates only 21. The threshold is doing real work.

**The measured gap decomposition.** The 33.4-point gap from the production filled-cell accuracy (66.6%) to a hypothetical 100% breaks down cleanly:

| Configuration | Filled acc | Gap contribution |
|---|---:|---:|
| Production `detect_grid` + 4-point warp | 66.6% | — (baseline) |
| GT corners + 4-point warp (perfect detection) | 78.5% | **+11.9 from detection quality** |
| GT corners + 8-point piecewise warp (perfect warp) | 84.7% | **+6.2 from interior warp** |
| Theoretical 100% | 100% | **+15.3 classifier + preprocessing ceiling** |

The largest single lever is detection (11.9 points), but that's essentially frozen — the 4-step `detect_grid` fallback chain is already on its 4th iteration. The *practically actionable* gap is the 6.2 points from the piecewise warp, plus whatever fraction of the 15.3-point classifier ceiling is reachable.

**The v6 retrain-then-rollback.** A 9-config architecture ablation (`evaluation/ablation.py`) identified a depth-4 / [32, 64, 128, 256] variant (405,898 params) as the winner at +5.2 filled-cell points over v5.1 — but that was measured on ground-truth corners (perfectly annotated cell crops). When the same architecture was retrained at full 30 epochs and evaluated on the production `detect_grid` path, it **regressed** to 65.2% filled (v5.1: 66.6%). The larger model was *more sensitive* to the imperfect cell crops from the production 4-point warp — its warp-quality tax was ~15 points vs v5.1's ~9 points. Bigger model + noisy inputs = worse, not better. v6 was rolled back to v5.1 within the same session.

**This confirmed what the gap decomposition predicted:** the classifier ceiling is real. Throwing 4x the parameters at the CNN didn't help because the bottleneck isn't model capacity — it's the quality of the cell crops arriving at the classifier's input. The per-image distribution is still **bimodal**:

- **15 of 38 images**: >=90% filled-cell accuracy. The pipeline is nearly perfect on these.
- **14 of 38 images**: 50-90% accuracy. Some errors, mostly recoverable manually.
- **9 of 38 images**: <50% accuracy. Extreme cases — heavily faded print, mixed handwritten/printed, extreme motion blur.

**Conclusion from the analysis:** further CNN retraining yields noise-level improvements at best. The real leverage is upstream: wiring the 8-point piecewise warp (`extract_cells_piecewise`) into `/api/extract` so that cells from curved newsprint arrive with cleaner boundaries. That's measured at +6.2 filled-cell points. That's why the main README's Roadmap says "wire piecewise warp" and not "train a bigger CNN."

## 7. ONNX deployment — the "24 KB" lie

The trained checkpoint is exported to ONNX and loaded via ONNX Runtime. The deployment requirements file (`requirements-deploy.txt`) is deliberately PyTorch-free. But the `.onnx` file itself is misleading:

```
app/ml/checkpoints/
├── sudoku_cnn.onnx         4,152 B  (~ 4 KB protobuf header)
└── sudoku_cnn.onnx.data  405,120 B  (~396 KB weight tensors, REQUIRED)
                          ─────────
                           ~400 KB total
```

The `.onnx` file is a thin protobuf header that references an external sidecar file holding the weight tensors. **Loading the `.onnx` alone — without its `.data` sidecar in the same directory — fails at ONNX Runtime session initialization.** I verified this by temporarily renaming the `.data` file during the 2026-04-10 audit.

An earlier version of the README claimed "24 KB ONNX" as a deployment headline. That was the protobuf header size from an older export (pre-2026-04-10 exports embedded more metadata, producing a ~24 KB header; the current v5.1 export via `app/ml/export_onnx.py` produces the leaner ~4 KB header). The real deployment footprint is ~400 KB. Both files are committed to git.

Moral: when you claim a deployment size, **load your model from a freshly cloned directory and see if it actually works**. The "24 KB" claim would have been immediately falsified if I'd done that once.

## 8. References

- `app/ml/model.py:24` — `SudokuCNN` architecture
- `app/ml/dataset.py` — `create_datasets` (MNIST 1-9 + printed 67-font + Chars74K font-disjoint + empty cells, with class-weighted loss)
- `app/ml/train.py` — training loop (30 epochs, Adam + CosineAnnealingLR, early stopping at patience 7, `CrossEntropyLoss(weight=[2, 1, ..., 1])`)
- `app/ml/recognizer.py:46` — ONNX Runtime wrapper used at inference time
- [`ocr_analysis.ipynb`](ocr_analysis.ipynb) — the per-cell error taxonomy and threshold sweep that drove the "stop tuning OCR, fix the warp" decision
- `app/ml/checkpoints/training_results.json` — training history, confusion matrix, final test accuracy
- `evaluation/ablation.py` + `evaluation/ablation_results.json` — 9-config architecture ablation (depth x width at dropout 0.3), which identified d4_c-medium as the GT-corners winner but whose lift did not transfer to the production `detect_grid` path (see the v6 rollback story in section 6)
- Kamal, Chawla & Goel, "Techniques for Solving Sudoku Puzzles" (ICIIP 2015) — the classical CSP framework that the backtracking solver follows; also informs the CNN's role as a thin recognition layer feeding a constraint-based solver rather than an end-to-end grid predictor